In [1]:
import torch

In [2]:
pert_map = torch.load("ST-HVG-Tahoe/fewshot/state_generalization_X_hvg/pert_onehot_map.pt", map_location="cpu", weights_only=False)

In [3]:
print(type(pert_map))

<class 'dict'>


In [4]:
first_key = next(iter(pert_map))

In [5]:
print(first_key, "->", pert_map[first_key])
print(f"Total entries: {len(pert_map)}")

[('(R)-Verapamil (hydrochloride)', 0.05, 'uM')] -> tensor([1., 0., 0.,  ..., 0., 0., 0.])
Total entries: 1138


In [6]:
print(f"Dimensions of the one-hot encoding: {pert_map[first_key].shape}")

Dimensions of the one-hot encoding: torch.Size([1138])


In [7]:
import pickle
with open("../../notebooks/jeannie/ST-HVG-Tahoe/fewshot/state_generalization_X_hvg/cell_type_onehot_map.pkl", 'rb') as file:
    cell_type_map = pickle.load(file)

print(type(cell_type_map))

<class 'dict'>


In [8]:
first_key = next(iter(cell_type_map))
print(first_key, "->", cell_type_map[first_key])
print(f"Total entries: {len(cell_type_map)}")

A-172 -> tensor([1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])
Total entries: 50


In [11]:
print(any("SW" in key.upper() for key in cell_type_map.keys()))

True


In [14]:
print(any("SW480" in key.upper() for key in cell_type_map.keys()))

True


In [15]:
cell_line = "SW480"
encoding = cell_type_map[cell_line]
print(cell_line, "->", encoding)

SW480 -> tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0.])


In [18]:
# grab the initial cell state in tahoe dataset where it's the desired cell type with the DMSO pert variant
dmso = "[('DMSO_TF', 0.0, 'uM')]"
print(pert_map[dmso].float().shape)

torch.Size([1138])


In [1]:
!pip install datasets

  Using cached datasets-5.0.0-py3-none-any.whl.metadata (23 kB)
  Using cached dill-0.4.1-py3-none-any.whl.metadata (10 kB)
  Using cached xxhash-3.8.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (14 kB)
  Using cached multiprocess-0.70.19-py312-none-any.whl.metadata (7.5 kB)
Using cached datasets-5.0.0-py3-none-any.whl (555 kB)
Using cached dill-0.4.1-py3-none-any.whl (120 kB)
Using cached multiprocess-0.70.19-py312-none-any.whl (150 kB)
Using cached xxhash-3.8.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl (220 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [datasets]3/4 [datasets]ess]


In [2]:
from datasets import load_dataset
tahoe_100m_ds = load_dataset("tahoebio/Tahoe-100M", streaming=True, split="train")

# tahoe_100m_ds.head()

/home/jeannie/miniconda/envs/pytorch-pip/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from datasets import load_dataset
import numpy as np

# --- 1. Resolve "SW480" -> Cellosaurus ID (expression_data keys on CVCL_ id, not cell name) ---
sw480_cvcl_id = "CVCL_0546"

# --- 2. Gene token_id -> gene_symbol map (needed to reconstruct dense vectors) ---
gene_meta = load_dataset("tahoebio/Tahoe-100M", "gene_metadata", split="train").to_pandas()
n_genes_total = gene_meta["token_id"].max() + 1
token_to_symbol = gene_meta.set_index("token_id")["gene_symbol"].to_dict()
gene_symbol_by_token = np.array(
    [token_to_symbol.get(i, "") for i in range(n_genes_total)]
)  # ordered array, index = token_id

# --- 3. Stream-filter expression_data for SW480 + DMSO_TF (vehicle control) ---
ds = load_dataset("tahoebio/Tahoe-100M", streaming=True, split="train")
matches = ds.filter(
    lambda row: row["cell_line_id"] == sw480_cvcl_id and row["drug"] == "DMSO_TF"
)

# --- 4. Pull cells and reconstruct dense, normalized profiles ---
def sparse_row_to_dense(row, n_genes_total):
    # First entry in both `genes` and `expressions` is a marker/CLS token — discard it.
    genes_tok_ids = np.asarray(row["genes"][1:])
    expressions = np.asarray(row["expressions"][1:], dtype=np.float32)
    dense = np.zeros(n_genes_total, dtype=np.float32)
    dense[genes_tok_ids] = expressions
    return dense

def normalize_log1p(dense_counts, target_sum=1e4):
    lib_size = dense_counts.sum()
    if lib_size == 0:
        return dense_counts
    return np.log1p(dense_counts / lib_size * target_sum)

N_CELLS = 50  # how many SW480/DMSO_TF cells to pull
profiles = []
for row in matches.take(N_CELLS):
    dense = sparse_row_to_dense(row, n_genes_total)
    profiles.append(normalize_log1p(dense))
profiles = np.stack(profiles)  # (N_CELLS, n_genes_total), full-panel normalized log1p

# --- 5. Subset to STATE's 2000-HVG panel ---
# STATE's HVG selection is a separate preprocessing artifact (from `state tx preprocess_train
# --num_hvgs 2000`), not something Tahoe-100M raw data exposes directly. Load the exact gene
# list/order your STATE preprocessing already produced and persisted — do NOT re-select HVGs
# independently here, or your reward classifier's feature space won't match STATE's X_hvg output.
state_hvg_genes = np.load("state_hvg_gene_list.npy", allow_pickle=True)  # <- your saved panel

symbol_to_col = {sym: i for i, sym in enumerate(gene_symbol_by_token)}
hvg_col_idx = [symbol_to_col[g] for g in state_hvg_genes if g in symbol_to_col]
missing = set(state_hvg_genes) - set(gene_symbol_by_token[hvg_col_idx])
if missing:
    print(f"warning: {len(missing)} HVG genes not found in Tahoe gene_metadata: {missing}")

X_hvg = profiles[:, hvg_col_idx]  # (N_CELLS, 2000) — same space as STATE's X_hvg

In [3]:
!pip list | grep anndata

anndata                  0.12.16


In [6]:
!pwd

/home/jeannie/relearn/notebooks/jeannie


In [ ]:
import anndata as ad

adata = ad.read_h5ad("processed.h5ad", backed="r") # backed="r" avoids loading full matrix into memory

In [ ]:

# Basic shape / contents
print(adata)                      # summary: n_obs x n_vars, obs/var/obsm/uns keys
print(adata.var.columns.tolist()) # look for 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'variances', 'variances_norm'
print(adata.var.head(20))

# The HVG flag and the actual selected gene list
hvg_mask = adata.var["highly_variable"]
print(f"n HVGs flagged: {hvg_mask.sum()}")
hvg_genes = adata.var_names[hvg_mask].tolist()

# X_hvg matrix itself
print(adata.obsm["X_hvg"].shape)  # should be (n_cells, 2000)

# Check uns for stored HVG params (flavor, batch_key, etc. — this is what answers your last question)
print(adata.uns.keys())
for k in adata.uns:
    if "hvg" in k.lower() or "highly_variable" in k.lower():
        print(k, ":", adata.uns[k])

# Cell-line / cell-type breakdown — confirm SW480 is in there and how HVGs relate to it
print(adata.obs["cell_type"].value_counts())  # or whatever the cell-line column is called — check adata.obs.columns first

: 

: 

In [1]:
import torch
dm = torch.load("ST-HVG-Tahoe/fewshot/state_generalization_X_hvg/data_module.torch", map_location="cpu", weights_only=False)
print(type(dm))
print(dir(dm))  # scan for anything like var_names, hvg_genes, gene_names, adata

# common attribute names to try:
for attr in ["var_names", "hvg_genes", "gene_names", "genes", "adata"]:
    if hasattr(dm, attr):
        print(attr, "->", getattr(dm, attr))

<class 'dict'>
['__class__', '__class_getitem__', '__contains__', '__delattr__', '__delitem__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getitem__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__ior__', '__iter__', '__le__', '__len__', '__lt__', '__ne__', '__new__', '__or__', '__reduce__', '__reduce_ex__', '__repr__', '__reversed__', '__ror__', '__setattr__', '__setitem__', '__sizeof__', '__str__', '__subclasshook__', 'clear', 'copy', 'fromkeys', 'get', 'items', 'keys', 'pop', 'popitem', 'setdefault', 'update', 'values']


In [2]:
dm.keys()

dict_keys(['toml_config_path', 'batch_size', 'num_workers', 'random_seed', 'pert_col', 'batch_col', 'cell_type_key', 'control_pert', 'embed_key', 'output_space', 'basal_mapping_strategy', 'n_basal_samples', 'should_yield_control_cells', 'cell_sentence_len', 'cache_perturbation_control_pairs', 'map_controls', 'perturbation_features_file', 'int_counts', 'normalize_counts', 'store_raw_basal', 'barcode'])

In [6]:
print(dm['toml_config_path'])

/data/tahoe_se/generalization.toml


In [ ]:
import anndata as ad

h5ad_path = "processed.h5ad"
adata = ad.read_h5ad(h5ad_path, backed="r")
hvg_genes = adata.var_names[adata.var["highly_variable"]].tolist()